# 07  Results summary and check against the midterm

**File:** `07_results_summary.ipynb`   |   **Run order:** last

Prints every result table in one place (the numbers the report and slides use) and checks that the split notebooks reproduce the frozen midterm results.

**Reads:** all tables in the results folder, and the frozen results in `outputs`

**Writes (into the results folder):** nothing

**To run:** run all cells from top to bottom. The settings come from `config.py` in this folder.

*The section below is the original text of this task from the combined notebook.*

In [ ]:
from config import *

# Results to copy back

The cell below prints every result table in one place. These are the numbers the report and the slides use.

In [62]:
def show(fname, title):
    p = OUT_DIR / fname
    print("=" * 100); print(title, "|", fname)
    if p.exists():
        print(pd.read_csv(p).to_string(index=False))
    else:
        print("(file not found)")

show("table_data_checks.csv", "Data checks")
show("predictor_accuracy.csv", "Accuracy of the predictors (test, validation)")
show("eval_summary_test.csv", "Pre-set cut-offs: summary")
show("eval_by_length_test.csv", "Pre-set cut-offs: recall by event length")
show("eval_equal_rate_summary_test.csv", "Equal alarm rates: summary")
show("eval_equal_rate_by_length_test.csv", "Equal alarm rates: recall by event length")
show("eval_equal_rate_test.csv", "Equal alarm rates: curve")
print()
print("Files in", OUT_DIR, ":")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name:45s} {p.stat().st_size/1e6:8.2f} MB")
print(f"\nTotal run time: {(time.time() - T_START)/60:.1f} min")

Data checks | table_data_checks.csv
                                     Unnamed: 0       value
                                      buildings     200.000
                                           rows 1749494.000
                           exact duplicate rows       0.000
              duplicate building-timestamp rows       0.000
                              negative readings       0.000
                               missing readings       0.000
                            exact zero readings   70157.000
                       share of exact zeros (%)       4.010
rows not exactly 1 h after the previous row (%)       0.074
                     row-based lag_24 wrong (%)       0.890
Accuracy of the predictors (test, validation) | predictor_accuracy.csv
split                        predictor  hours  MAE_kWh  RMSE_kWh    R2  RMSLE
  val                Same as last hour 249407     9.10     49.09 0.984  0.234
  val         Historical typical value 249407    23.18     65.28 0.971  0.753

In [ ]:
# ---- Check: do these results reproduce the frozen midterm results in "outputs"? ----
# Two kinds of file, two kinds of check:
#   EXACT  = depends only on the data and the simple rules (no trained model). Must match to rounding.
#   MODEL  = depends on the three trained models A, B, C. A tree model trained again, with another
#            library version, gives slightly different numbers, so these may be "close" but not identical.
# "close" means: every number is within 5% or within an allowance for small counts, where the allowance
#                is set per file (below), AND the ranking of the methods at the 1% flag rate is unchanged.
#                Tables split by event length have groups of only 30 to 250 events, so one event is
#                1 to 3 percentage points; a retrained model can move a group by two or three events.
import json
EXACT = ["table_data_checks.csv", "table_events_usable.csv"]
MODEL = {"predictor_accuracy.csv": 1.0, "eval_summary_test.csv": 3.0, "eval_by_length_test.csv": 7.0,
         "eval_equal_rate_summary_test.csv": 1.0, "eval_equal_rate_by_length_test.csv": 7.0, "eval_equal_rate_test.csv": 7.0}
REL_TOL, ABS_TOL = 0.05, 1.0     # ABS_TOL is the default; the MODEL dictionary gives the allowance per file

def compare(name, kind, abs_tol=ABS_TOL):
    a, b = OUT_DIR / name, MIDTERM_DIR / name
    if not a.exists() or not b.exists():
        return ("file missing in " + ("results" if not a.exists() else "midterm")), np.nan
    x, y = pd.read_csv(a), pd.read_csv(b)
    if x.shape != y.shape or list(x.columns) != list(y.columns):
        return "DIFFERENT (shape or columns)", np.nan
    num = x.select_dtypes("number").columns
    xv, yv = x[num].to_numpy(float), y[num].to_numpy(float)
    if not x.drop(columns=num).equals(y.drop(columns=num)):
        return "DIFFERENT (text columns)", np.nan
    if np.allclose(xv, yv, rtol=1e-6, atol=1e-9, equal_nan=True):
        return "match", 0.0
    diff = np.abs(xv - yv)
    worst = float(np.nanmax(diff)) if diff.size else 0.0
    ok = np.all((diff <= np.maximum(REL_TOL * np.abs(yv), abs_tol)) | np.isnan(diff))
    if kind == "EXACT":
        return "DIFFERENT", worst
    return ("close" if ok else "DIFFERENT"), worst

rows = []
for n in EXACT:
    r, w = compare(n, "EXACT"); rows.append((n, "exact", r, w))
for n, tol in MODEL.items():
    r, w = compare(n, "MODEL", tol); rows.append((n, "model", r, w))

# cut-offs
try:
    ca = json.load(open(OUT_DIR / "cutoffs.json"))["cutoffs"]; cb = json.load(open(MIDTERM_DIR / "cutoffs.json"))["cutoffs"]
    exact_keys = [k for k in cb if k in ("last_hour", "typical")]
    model_keys = [k for k in cb if k not in exact_keys]
    e_ok = all(abs(ca[k] - cb[k]) < 1e-3 for k in exact_keys)   # 1e-3: rounding when predictions go through a csv.gz file instead of parquet
    rows.append(("cutoffs.json: last_hour, typical", "exact", "match" if e_ok else "DIFFERENT", max(abs(ca[k]-cb[k]) for k in exact_keys)))
    worst = max(abs(ca[k] - cb[k]) for k in model_keys)
    m_exact = worst < 1e-6
    m_close = all(abs(ca[k] - cb[k]) <= max(REL_TOL * abs(cb[k]), ABS_TOL) for k in model_keys)
    rows.append(("cutoffs.json: A, B, C", "model", "match" if m_exact else ("close" if m_close else "DIFFERENT"), worst))
except Exception as e:
    rows.append(("cutoffs.json", "-", "could not compare: " + str(e), np.nan))

# ranking of the methods at 1% flagged, in both modes
try:
    s1 = pd.read_csv(OUT_DIR / "eval_equal_rate_summary_test.csv"); s0 = pd.read_csv(MIDTERM_DIR / "eval_equal_rate_summary_test.csv")
    col = [c for c in s1.columns if c.startswith("recall_at_")][0]
    same_rank = all(list(s1[s1["mode"] == m].sort_values(col, ascending=False)["method"]) ==
                    list(s0[s0["mode"] == m].sort_values(col, ascending=False)["method"]) for m in s0["mode"].unique())
    rows.append(("ranking of the six methods at 1% flagged", "model", "same ranking" if same_rank else "RANKING CHANGED", np.nan))
except Exception as e:
    rows.append(("ranking at 1%", "-", "could not compare: " + str(e), np.nan))

out = pd.DataFrame(rows, columns=["file", "kind", "result", "largest difference"])
print(out.to_string(index=False))
print()
bad = out["result"].str.startswith(("DIFFERENT", "RANKING", "file missing", "could not"))
if bad.any():
    print("STOP: at least one row is not a match or close. Check before building on these notebooks.")
elif (out["result"] == "close").any():
    print("OK: everything that depends only on the data matches exactly; the trained-model results are close to the midterm and rank the methods the same way.")
else:
    print("OK: every row matches.")
